# Analyse Exploratoire des Données — Kaggle Credit Card Fraud Detection

Ce notebook réalise l'analyse exploratoire du jeu de données Kaggle Credit Card Fraud Detection (ULB).
Dataset: 284 807 transactions, 30 features (V1-V28 + Amount + Time), cible: Class (0/1).

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12, 'figure.dpi': 300, 'font.family': 'serif'})
sns.set_style('whitegrid')

FIGURES_DIR = os.path.join('..', 'reports', 'figures', 'eda')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Load dataset
try:
    from src.data.loader import load_creditcard, get_dataset_info
    df = load_creditcard()
except FileNotFoundError:
    print("Dataset non trouvé. Téléchargez creditcard.csv dans data/raw/")
    print("https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud")
    df = None

In [ ]:
if df is not None: print(f"Shape: {df.shape}"); display(df.head()); print(f"\nInfo:"); print(df.info())

## Statistiques Descriptives

In [ ]:
if df is not None: display(df.describe().round(2))

## Distribution des Classes

Le déséquilibre est extrême: seulement 0.173% de fraudes. Ce ratio de 578:1 constitue le défi principal.

In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    counts = df['Class'].value_counts()
    colors = ['#2ecc71', '#e74c3c']
    axes[0].bar(['Légitime\n(n={:,})'.format(counts[0]), 'Fraude\n(n={:,})'.format(counts[1])],
                counts.values, color=colors, edgecolor='black', alpha=0.8)
    axes[0].set_ylabel('Nombre de Transactions')
    axes[0].set_title('Distribution des Classes')
    
    # Pie chart
    axes[1].pie(counts.values, labels=['Légitime', 'Fraude'], colors=colors,
                autopct='%1.3f%%', startangle=90, explode=(0, 0.1))
    axes[1].set_title('Proportion des Classes')
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'class_distribution.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Légitime: {counts[0]:,} ({counts[0]/len(df)*100:.3f}%)")
    print(f"Fraude: {counts[1]:,} ({counts[1]/len(df)*100:.3f}%)")
    print(f"Ratio d'imbalance: {counts[0]/counts[1]:.0f}:1")

## Distribution des Montants

Nous observons que les montants frauduleux ne sont pas systématiquement élevés — la moyenne est modeste (~€122).

In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Linear scale
    axes[0].hist(df[df['Class']==0]['Amount'], bins=50, alpha=0.7, label='Légitime', color='#2ecc71', density=True)
    axes[0].hist(df[df['Class']==1]['Amount'], bins=50, alpha=0.7, label='Fraude', color='#e74c3c', density=True)
    axes[0].set_xlabel('Montant (€)')
    axes[0].set_ylabel('Densité')
    axes[0].set_title('Distribution des Montants')
    axes[0].legend()
    axes[0].set_xlim(0, 1000)
    
    # Log scale
    axes[1].hist(np.log1p(df[df['Class']==0]['Amount']), bins=50, alpha=0.7, label='Légitime', color='#2ecc71', density=True)
    axes[1].hist(np.log1p(df[df['Class']==1]['Amount']), bins=50, alpha=0.7, label='Fraude', color='#e74c3c', density=True)
    axes[1].set_xlabel('log(1 + Montant)')
    axes[1].set_ylabel('Densité')
    axes[1].set_title('Distribution des Montants (échelle log)')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'amount_histogram.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    
    print(f"Montant moyen (légitime): €{df[df['Class']==0]['Amount'].mean():.2f}")
    print(f"Montant moyen (fraude): €{df[df['Class']==1]['Amount'].mean():.2f}")
    print(f"Montant max (fraude): €{df[df['Class']==1]['Amount'].max():.2f}")

## Distribution Temporelle

Les transactions suivent un cycle circadien. Les fraudes sont plus uniformément distribuées.

In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(12, 5))
    hours = df['Time'] / 3600
    
    ax.hist(hours[df['Class']==0], bins=48, alpha=0.6, label='Légitime', color='#2ecc71', density=True)
    ax.hist(hours[df['Class']==1], bins=48, alpha=0.6, label='Fraude', color='#e74c3c', density=True)
    ax.set_xlabel('Temps (heures depuis première transaction)')
    ax.set_ylabel('Densité')
    ax.set_title('Distribution Temporelle des Transactions')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'time_distribution.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

## Matrice de Corrélation

Les features V1-V28, issues de l'ACP, sont par construction quasi-orthogonales.

In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(16, 14))
    corr = df.drop(columns=['Class']).corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, ax=ax,
                square=True, linewidths=0.3, cbar_kws={'shrink': 0.8})
    ax.set_title('Matrice de Corrélation des Features')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'correlation_heatmap.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

## Features Discriminantes

V14, V4, V12 et V10 présentent les distributions les plus différentes entre fraude et légitime.

In [ ]:
if df is not None:
    top_features = ['V14', 'V4', 'V12', 'V10', 'V17', 'V3']
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    for idx, feat in enumerate(top_features):
        ax = axes[idx // 3, idx % 3]
        data_legit = df[df['Class']==0][feat]
        data_fraud = df[df['Class']==1][feat]
        ax.boxplot([data_legit, data_fraud], labels=['Légitime', 'Fraude'],
                   patch_artist=True, boxprops=dict(facecolor='lightblue'))
        ax.set_title(f'{feat}')
        ax.grid(True, alpha=0.3)
    
    fig.suptitle('Distribution des Features Discriminantes (Fraude vs Légitime)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'feature_boxplots.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

## KDE des Features Principales

In [ ]:
if df is not None:
    top_features = ['V14', 'V4', 'V12', 'V10', 'V17', 'V3']
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    for idx, feat in enumerate(top_features):
        ax = axes[idx // 3, idx % 3]
        df[df['Class']==0][feat].plot.kde(ax=ax, label='Légitime', color='#2ecc71')
        df[df['Class']==1][feat].plot.kde(ax=ax, label='Fraude', color='#e74c3c')
        ax.set_title(f'Distribution de {feat}')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    fig.suptitle('Density Plots — Top 6 Features Discriminantes', fontsize=14)
    plt.tight_layout()
    plt.show()

## Comparaison Statistique par Classe

In [ ]:
if df is not None:
    means_legit = df[df['Class']==0].drop(columns=['Class']).mean()
    means_fraud = df[df['Class']==1].drop(columns=['Class']).mean()
    comparison = pd.DataFrame({'Légitime': means_legit, 'Fraude': means_fraud})
    comparison['Écart Absolu'] = abs(comparison['Fraude'] - comparison['Légitime'])
    comparison = comparison.sort_values('Écart Absolu', ascending=False)
    display(comparison.round(4).head(15))
    print(f"\nFeatures les plus discriminantes: {list(comparison.head(6).index)}")

## Résumé des Observations

1. **Déséquilibre extrême**: ratio 578:1, nécessite un traitement spécifique (SMOTE, cost-sensitive)
2. **Montants**: les fraudes ne sont pas systématiquement à montant élevé (moyenne ~€122 vs ~€88)
3. **Temporalité**: distribution plus uniforme des fraudes, légère concentration nocturne
4. **Features discriminantes**: V14, V4, V12, V10, V17 montrent les plus grandes différences
5. **Corrélation**: features PCA quasi-orthogonales, peu de multicolinéarité
6. **Pas de valeurs manquantes**: dataset propre, prêt pour la modélisation